# Testing the Uni Mainz KI-Chat API

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yfeng-hsm/KI_Geodatenanalyse_SS26/blob/main/lectures/03_llm_basics/notebooks/uni_mainz_ki_chat_api_test.ipynb)

This notebook tests the OpenAI-compatible KI-Chat@JGU API provided by ZDV Mainz.

Official documentation: https://www.zdv.uni-mainz.de/ki-chat-api-nutzung/

The API base URL is:

```text
https://ki-chat.uni-mainz.de/api
```

The notebook covers:

1. Entering an API key manually for the current notebook session
2. Listing available models with `/models`
3. Sending a small `/chat/completions` request
4. Optionally testing `/embeddings`
5. Showing how to handle common HTTP errors

Do not write API keys into saved notebook cells before committing or sharing the file.

## 1. Setup

Create an API key in the KI-Chat@JGU web interface:

1. Log in at https://ki-chat.uni-mainz.de
2. Open the avatar menu in the top right
3. Go to settings and then account
4. Create a new API key
5. Store it securely

Run the next cell every time you start a fresh notebook session. It asks for the API key manually and hides the input.

In [ ]:
from __future__ import annotations

import getpass
import json
import textwrap
from typing import Any

import requests
from IPython.display import Markdown, display

API_BASE_URL = "https://ki-chat.uni-mainz.de/api"
API_KEY = getpass.getpass("Paste your KI-Chat@JGU API key for this session: ").strip()

if not API_KEY:
    raise RuntimeError("No API key entered. Run this cell again and paste your KI-Chat@JGU API key.")

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

print("API key loaded for this notebook session.")
print(f"API base URL: {API_BASE_URL}")

## 2. Helper Function

The Uni Mainz endpoint follows the OpenAI-style HTTP API. The helper below keeps requests small and prints useful error information when something fails.

In [ ]:
def api_request(method: str, endpoint: str, **kwargs: Any) -> Any:
    """Call the KI-Chat API and return parsed JSON."""
    url = f"{API_BASE_URL}{endpoint}"
    response = requests.request(method, url, headers=headers, timeout=60, **kwargs)

    if not response.ok:
        print(f"HTTP {response.status_code} for {method} {endpoint}")
        try:
            print(json.dumps(response.json(), indent=2, ensure_ascii=False))
        except ValueError:
            print(response.text[:1000])
        response.raise_for_status()

    return response.json()


def preview_json(data: Any, max_chars: int = 3000) -> None:
    """Pretty-print JSON without flooding the notebook."""
    text = json.dumps(data, indent=2, ensure_ascii=False)
    if len(text) > max_chars:
        text = text[:max_chars] + "\n..."
    print(text)

## 3. List Available Models

The `/models` endpoint returns the model names currently available to your account. Use one of those names in the chat request below.

In [ ]:
models_response = api_request("GET", "/models")
preview_json(models_response)

In [ ]:
models = models_response.get("data", [])
model_ids = [model.get("id") for model in models if model.get("id")]

print("Available model IDs:")
for model_id in model_ids:
    print("-", model_id)

DEFAULT_CHAT_MODEL = "GPT OSS 120B"
chat_model = DEFAULT_CHAT_MODEL if DEFAULT_CHAT_MODEL in model_ids else (model_ids[0] if model_ids else DEFAULT_CHAT_MODEL)

print(f"\nUsing chat model: {chat_model}")

## 4. Chat Completion Test

This sends a short geospatial-analysis prompt. Keep tests small to respect the documented rate limits.

In [ ]:
payload = {
    "model": chat_model,
    "messages": [
        {
            "role": "system",
            "content": "You are a concise teaching assistant for geospatial data analysis.",
        },
        {
            "role": "user",
            "content": "Explain in 3 bullet points how an LLM can help inspect OpenStreetMap data quality.",
        },
    ],
    "temperature": 0.2,
    "max_tokens": 300,
}

chat_response = api_request("POST", "/chat/completions", json=payload)
preview_json(chat_response)

In [ ]:
message = chat_response["choices"][0]["message"]["content"]
print(textwrap.fill(message, width=100))

## 5. Optional: Embeddings Test

The documentation lists `bge-m3` as the embedding model. Embeddings are useful for semantic search, clustering, and retrieval-augmented generation.

In [ ]:
embedding_payload = {
    "model": "bge-m3",
    "input": [
        "Mainz has dense cycling infrastructure near the university.",
        "A spatial join can attach census attributes to grid cells.",
    ],
}

embedding_response = api_request("POST", "/embeddings", json=embedding_payload)

vectors = [item["embedding"] for item in embedding_response.get("data", [])]
print(f"Received {len(vectors)} embedding vectors.")
if vectors:
    print(f"Embedding dimension: {len(vectors[0])}")
    print("First 8 values of the first vector:", vectors[0][:8])

## 6. Optional: Streaming Chat Completion

Set `stream=True` to receive incremental deltas. This cell uses `requests` directly because streamed responses are sent line by line.

In [ ]:
stream_payload = {
    "model": chat_model,
    "messages": [
        {
            "role": "user",
            "content": (
                "Explain spatial autocorrelation in exactly 10 clear sentences. "
                "Use a geospatial data analysis example."
            ),
        }
    ],
    "stream": True,
    # Increase this if you ask for longer explanations.
    "max_tokens": 900,
}

chunks = []

with requests.post(
    f"{API_BASE_URL}/chat/completions",
    headers=headers,
    json=stream_payload,
    stream=True,
    timeout=60,
) as response:
    response.raise_for_status()
    for line in response.iter_lines(decode_unicode=True):
        if not line or not line.startswith("data: "):
            continue
        data = line.removeprefix("data: ")
        if data == "[DONE]":
            break
        chunk = json.loads(data)
        delta = chunk["choices"][0].get("delta", {})
        token = delta.get("content", "")
        if token:
            chunks.append(token)
            print(token, end="", flush=True)

full_answer = "".join(chunks)
print("\n\n--- Full answer with wrapping ---")
display(Markdown(full_answer))


## 7. Optional: Multimodal Street Image Test

The ZDV documentation lists `Qwen3 235B VL` as a multimodal vision-language model. This test downloads a public Wikimedia street image, converts it to a base64 data URL, sends it to the model, and asks the model to describe visible street features.

Default image URL:

```text
https://upload.wikimedia.org/wikipedia/commons/f/f1/Mapillary_Z%C3%BCrich_1640896079862130_%28u3iVcplwqT9IFZXPjQEbyA_by_fcignat_%E2%80%93_2025-03-10_05H39M21S366%29.jpg
```

You can replace `STREET_IMAGE_URL` with another public image URL if needed.


In [ ]:
import base64

from IPython.display import Image as IPythonImage

STREET_IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/f/f1/Mapillary_Z%C3%BCrich_1640896079862130_%28u3iVcplwqT9IFZXPjQEbyA_by_fcignat_%E2%80%93_2025-03-10_05H39M21S366%29.jpg"

image_response = requests.get(STREET_IMAGE_URL, timeout=60)
image_response.raise_for_status()

street_image_mime = image_response.headers.get("Content-Type", "image/jpeg").split(";", 1)[0]
street_image_base64 = base64.b64encode(image_response.content).decode("utf-8")

print(f"Downloaded street image as {street_image_mime}")
display(IPythonImage(url=STREET_IMAGE_URL, width=700))


In [ ]:
vision_model = "Qwen3 235B VL"
if "model_ids" in globals() and model_ids and vision_model not in model_ids:
    print(f"Warning: {vision_model!r} was not listed by /models. Run the request anyway, or set vision_model manually.")

vision_payload = {
    "model": vision_model,
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "Describe what you see in this street image in exactly 10 sentences. "
                        "Mention transport infrastructure, land use, people or vehicles if visible, "
                        "and any safety-relevant observations."
                    ),
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:{street_image_mime};base64,{street_image_base64}"
                    },
                },
            ],
        }
    ],
    "temperature": 0.2,
    "max_tokens": 1000,
}

vision_response = api_request("POST", "/chat/completions", json=vision_payload)
vision_answer = vision_response["choices"][0]["message"]["content"]
display(Markdown(vision_answer))


## 8. Common Errors

- `401 Unauthorized`: the API key is missing, expired, or copied incorrectly.
- `404 Not Found`: check that the endpoint starts with `https://ki-chat.uni-mainz.de/api`.
- `429 Too Many Requests`: reduce request frequency or parallel calls.
- Model error: run the `/models` cell again and use an available model ID.

The ZDV page documents these relevant limits: one long-running API request per second, at most two parallel chat-completion requests, and at most one parallel embeddings request.